# 14 · Serving check analysis: does the served path cost accuracy? (Step 5.1, D-063)

**Runtime: CPU** (Runtime → Change runtime type → **CPU**). No GPU, no servers: it re-reads the replay files notebook 13 saved on Drive (~3 min).

Notebook 13 showed served answers agree with offline ones on 92% of val questions (the D-060 bar was 95%). This measures whether the differences change **accuracy**: answering accuracy, answerability AP and quality macro-F1, served vs offline on the same photos, with paired confidence intervals. It also reports the best threshold without the quality rule.

## 1 · Drive, code, analysis

In [ ]:
import json
import os
import subprocess

from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
RUNS = f"{DRIVE_ROOT}/runs"
OUT = f"{RUNS}/serve-check"


def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout + r.stderr)
    if r.returncode:
        raise RuntimeError("Command failed. Stop here and paste the output above.")


drive.mount("/content/drive")
for f in ("replay_val.jsonl", "replay_test.jsonl"):
    assert os.path.exists(f"{OUT}/{f}"), f"{f} not found: run notebook 13 first"
sh(f"rm -rf {REPO_DIR} && git clone -q {REPO_URL} {REPO_DIR} && git -C {REPO_DIR} log --oneline -1")
sh(f"pip install -q -e {REPO_DIR}")
sh(
    f"python -m vizwiz.serve.tune --records {DATA}/processed --val-replay {OUT}/replay_val.jsonl --offline-val {RUNS}/e5-multi-lora-val --test-replay {OUT}/replay_test.jsonl --report {OUT}/tune_report.json"
)

rep = json.load(open(f"{OUT}/tune_report.json"))
par = rep["parity_val"]
print("ANSWER:", {k: par["answer"][k] for k in ("n", "same_answer_pct", "vqa_accuracy")})
print(
    "ANSWERABILITY:",
    {k: par["answerability"][k] for k in ("n", "same_decision_at_0.5_pct", "average_precision")},
)
print("QUALITY:", {k: par["quality"][k] for k in ("n", "same_labels_pct", "macro_f1")})
print("CHOSEN THRESHOLD:", rep["chosen_threshold"])
b = rep["val"]["best_without_quality_rule"]
print(
    "VAL BEST WITHOUT QUALITY RULE:",
    {
        k: b[k]
        for k in (
            "threshold",
            "vizwiz_accuracy",
            "answered_pct",
            "wrong_answers_pct_of_all",
            "wrongly_declined_pct_of_answerable",
        )
    },
)

**Paste back:** the commit line and the five lines starting `ANSWER`, `ANSWERABILITY`, `QUALITY`, `CHOSEN THRESHOLD`, `VAL BEST WITHOUT QUALITY RULE` (numbers only).

**Upload:** `MyDrive/vizwiz/runs/serve-check/tune_report.json` (it replaces the earlier one).

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")